# 评估示例：推送通知摘要器监控

评估（Evals）是**面向任务**且迭代式的，它们是检查和改进 LLM 集成表现的最佳方式。

在下面的评估中，我们将聚焦于**检测提示词变更是否引入回归**这一任务。

我们的用例是：
1. 我们一直通过在生产环境的聊天补全请求中设置 `store=True` 来记录聊天补全请求。注意，你也可以在管理面板中启用“默认开启”的日志记录 (https://platform.openai.com/settings/organization/data-controls/data-retention).
2. 我们想查看提示词变更是否引入了回归。

## 评估结构

评估包含两部分：“Eval”和“Run”。“Eval”保存测试标准的配置以及“Run”的数据结构。一个 Eval 可以有多个 Run，每个 Run 都会使用你的测试标准进行评估。

In [1]:
from openai import AsyncOpenAI
import os
import asyncio

os.environ["OPENAI_API_KEY"] = os.environ.get("OPENAI_API_KEY", "your-api-key")
client = AsyncOpenAI()

## 用例

我们要测试的集成是一个推送通知摘要功能，它接收多条推送通知并将其合并为一条，这是一次聊天补全调用。

# 生成测试数据

我将使用两个不同版本的提示词生成模拟的生产环境聊天补全请求，以测试各自的表现。第一个是“好”提示词，第二个是“坏”提示词。它们会带有不同的元数据，我们稍后会用到。

In [2]:
push_notification_data = [
        """
- New message from Sarah: "Can you call me later?"
- Your package has been delivered!
- Flash sale: 20% off electronics for the next 2 hours!
""",
        """
- Weather alert: Thunderstorm expected in your area.
- Reminder: Doctor's appointment at 3 PM.
- John liked your photo on Instagram.
""",
        """
- Breaking News: Local elections results are in.
- Your daily workout summary is ready.
- Check out your weekly screen time report.
""",
        """
- Your ride is arriving in 2 minutes.
- Grocery order has been shipped.
- Don't miss the season finale of your favorite show tonight!
""",
        """
- Event reminder: Concert starts at 7 PM.
- Your favorite team just scored!
- Flashback: Memories from 3 years ago.
""",
        """
- Low battery alert: Charge your device.
- Your friend Mike is nearby.
- New episode of "The Tech Hour" podcast is live!
""",
        """
- System update available.
- Monthly billing statement is ready.
- Your next meeting starts in 15 minutes.
""",
        """
- Alert: Unauthorized login attempt detected.
- New comment on your blog post: "Great insights!"
- Tonight's dinner recipe: Pasta Primavera.
""",
        """
- Special offer: Free coffee with any breakfast order.
- Your flight has been delayed by 30 minutes.
- New movie release: "Adventures Beyond" now streaming.
""",
        """
- Traffic alert: Accident reported on Main Street.
- Package out for delivery: Expected by 5 PM.
- New friend suggestion: Connect with Emma.
"""]

In [ ]:
PROMPTS = [
    (
        """
        You are a helpful assistant that summarizes push notifications.
        You are given a list of push notifications and you need to collapse them into a single one.
        Output only the final summary, nothing else.
        """,
        "v1"
    ),
    (
        """
        You are a helpful assistant that summarizes push notifications.
        You are given a list of push notifications and you need to collapse them into a single one.
        The summary should be longer than it needs to be and include more information than is necessary.
        Output only the final summary, nothing else.
        """,
        "v2"
    )
]

tasks = []
for notifications in push_notification_data:
    for (prompt, version) in PROMPTS:
        tasks.append(client.chat.completions.create(
            model="gpt-4o-mini",
            messages=[
                {"role": "developer", "content": prompt},
                {"role": "user", "content": notifications},
            ],
            store=True,
            metadata={"prompt_version": version, "usecase": "push_notifications_summarizer"},
        ))
await asyncio.gather(*tasks)

你可以在 https://platform.openai.com/logs. 查看刚刚创建的补全。

**请确保这些聊天补全已经显示出来，因为下一步需要用到它们。**

In [ ]:
completions = await client.chat.completions.list()
assert completions.data, "No completions found. You may need to enable logs in your admin panel."
completions.data[0]

# 设置评估

Eval 保存在多个 *Run* 之间共享的配置，它包含两个组成部分：
1. 数据源配置 `data_source_config`：未来的 *Run* 所遵循的 schema（列）。
    - `data_source_config` 使用 JSON Schema 来定义 Eval 中可用的变量。
2. 测试标准 `testing_criteria`：用于判断你的集成在数据源的每一*行*上是否正常工作。

在这个用例中，我们使用的是已存储的补全（stored completions），因此我们将设置相应的 data_source_config

**重要**
你可能会有许多不同的已存储补全用例，元数据是在评估中跟踪它们的最佳方式，可以让评估保持聚焦并面向任务。

In [5]:
# We want our input data to be available in our variables, so we set the item_schema to
# PushNotifications.model_json_schema()
data_source_config = {
    "type": "stored_completions",
    "metadata": {
        "usecase": "push_notifications_summarizer"
    }
}

这个 data_source_config 定义了整个评估中可用的变量。

已存储补全配置提供了两个可在整个评估中使用的变量：
1. {{item.input}}：发送给补全调用的消息
2. {{sample.output_text}}：助手返回的文本响应

**现在，我们将使用这些变量来设置测试标准。**

In [6]:
GRADER_DEVELOPER_PROMPT = """
Label the following push notification summary as either correct or incorrect.
The push notification and the summary will be provided below.
A good push notificiation summary is concise and snappy.
If it is good, then label it as correct, if not, then incorrect.
"""
GRADER_TEMPLATE_PROMPT = """
Push notifications: {{item.input}}
Summary: {{sample.output_text}}
"""
push_notification_grader = {
    "name": "Push Notification Summary Grader",
    "type": "label_model",
    "model": "o3-mini",
    "input": [
        {
            "role": "developer",
            "content": GRADER_DEVELOPER_PROMPT,
        },
        {
            "role": "user",
            "content": GRADER_TEMPLATE_PROMPT,
        },
    ],
    "passing_labels": ["correct"],
    "labels": ["correct", "incorrect"],
}

`push_notification_grader` 是一个模型评分器（LLM 作为评判者，llm-as-a-judge），它查看输入 `{{item.input}}` 和生成的摘要 `{{sample.output_text}}`，并将其标记为“correct”或“incorrect”。

注意：在底层，它使用结构化输出，因此标签始终有效。

**现在来创建我们的评估，并开始向其中添加数据！**

In [7]:
eval_create_result = await client.evals.create(
    name="Push Notification Completion Monitoring",
    metadata={"description": "This eval monitors completions"},
    data_source_config=data_source_config,
    testing_criteria=[push_notification_grader],
)

eval_id = eval_create_result.id

# 创建 Run

现在评估和 test_criteria 都已设置好，我们可以开始添加 Run 了。
我想比较两个**提示词版本**之间的表现

为此，我们只需将数据源定义为“stored_completions”，并为每个提示词版本设置元数据过滤器。

In [ ]:
# Grade prompt_version=v1
eval_run_result = await client.evals.runs.create(
    eval_id=eval_id,
    name="v1-run",
    data_source={
        "type": "completions",
        "source": {
            "type": "stored_completions",
            "metadata": {
                "prompt_version": "v1",
            }
        }
    }
)
print(eval_run_result.report_url)

In [ ]:
# Grade prompt_version=v2
eval_run_result_v2 = await client.evals.runs.create(
    eval_id=eval_id,
    name="v2-run",
    data_source={
        "type": "completions",
        "source": {
            "type": "stored_completions",
            "metadata": {
                "prompt_version": "v2",
            }
        }
    }
)
print(eval_run_result_v2.report_url)

为了更全面，我们来看看以两个提示词版本为起点，换用 4o 而不是 4o-mini 时，这个提示词的表现如何。

我们只需引用输入消息（{{item.input}}）并将模型设置为 4o。由于我们还没有任何 4o 的已存储补全，这次评估 Run 将生成新的补全。

In [ ]:
tasks = []
for prompt_version in ["v1", "v2"]:
    tasks.append(client.evals.runs.create(
        eval_id=eval_id,
        name=f"post-fix-new-model-run-{prompt_version}",
        data_source={
            "type": "completions",
            "input_messages": {
                "type": "item_reference",
                "item_reference": "item.input",
            },
            "model": "gpt-4o",
            "source": {
                "type": "stored_completions",
                "metadata": {
                    "prompt_version": prompt_version,
                }
            }
        },
    ))
result = await asyncio.gather(*tasks)
for run in result:
    print(run.report_url)

查看该报告，你会发现 prompt_version=v2 出现了回归！

## 恭喜，你刚刚发现了一个 bug，你可以回滚它，或者再修改提示词等等！